In [ ]:
# ==========================
# 0) LIBRARIES
# ==========================
import os
files = ['train.csv', 'test.csv', 'sample_prediction.csv']
if all(os.path.exists(f) for f in files):
    print('Ready!')
else:
    print('Upload files!')

import sys

!{sys.executable} -m pip -q install xgboost lightgbm

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_percentage_error, r2_score
import lightgbm as lgb
from lightgbm import LGBMRegressor

SEED = 42
np.random.seed(SEED)

In [ ]:
# =========================
# LOAD DATA
# =========================

train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sample = pd.read_csv("sample_prediction.csv")

print(train.shape, test.shape, sample.shape)
train.head()

In [ ]:
# =========================
# CLEAN + NORMALIZE (Friend's approach) - FIXED
# =========================
def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def drop_index_cols(df):
    df = df.copy()
    for c in ["Unnamed: 0", "index"]:
        if c in df.columns and df[c].nunique(dropna=False) == len(df):
            df = df.drop(columns=[c])
    return df

def normalize_cats(df):
    df = df.copy()
    for c in ["Status", "Sensor Position Side"]:
        if c in df.columns:
            s = df[c].astype("string").str.strip().str.lower()
            s = s.replace({"nan": pd.NA, "none": pd.NA, "": pd.NA})
            df[c] = s
    return df

train = normalize_cols(train)
test  = normalize_cols(test)

train = drop_index_cols(train)
test  = drop_index_cols(test)

train = normalize_cats(train)
test  = normalize_cats(test)

train = train.drop_duplicates()
print("After clean:", train.shape, test.shape)

In [ ]:
# =========================
# FEATURE ENGINEERING (Friend's safe_div approach) - FIXED
# =========================
import re

def safe_div(a, b):
    b = np.where(b == 0, np.nan, b)
    return a / b

def sanitize_colnames(cols):
    cols = [str(c) for c in cols]
    cols = [re.sub(r'[^0-9a-zA-Z_]+', '_', c) for c in cols]   # remove special chars
    cols = [re.sub(r'_+', '_', c).strip('_') for c in cols]   # collapse underscores
    return cols

def fe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Strong column sanitization (prevents LightGBM JSON feature-name error)
    df.columns = sanitize_colnames(df.columns)

    def has(cols):
        return all(c in df.columns for c in cols)

    # Tank volume + shape
    if has(["Tank_Width_m", "Tank_Length_m", "Tank_Height_m"]):
        W = df["Tank_Width_m"].astype(float)
        L = df["Tank_Length_m"].astype(float)
        H = df["Tank_Height_m"].astype(float)
        df["tank_vol"] = W * L * H
        df["tank_w_over_l"] = safe_div(W, L)

    # Vapour fraction
    if has(["Vapour_Height_m", "Tank_Height_m"]):
        df["vapour_height_frac"] = safe_div(
            df["Vapour_Height_m"].astype(float),
            df["Tank_Height_m"].astype(float)
        )

    # Obstacle area
    if has(["Obstacle_Width_m", "Obstacle_Height_m"]):
        df["obs_area"] = (
            df["Obstacle_Width_m"].astype(float) *
            df["Obstacle_Height_m"].astype(float)
        )

    # Sensor radius
    if has(["Sensor_Position_x", "Sensor_Position_y", "Sensor_Position_z"]):
        x = df["Sensor_Position_x"].astype(float)
        y = df["Sensor_Position_y"].astype(float)
        z = df["Sensor_Position_z"].astype(float)
        df["sensor_r"] = np.sqrt(x*x + y*y + z*z)

    # Sensor_r / obstacle distance
    if has(["sensor_r", "Obstacle_Distance_to_BLEVE_m"]):
        df["sensor_r_over_obsdist"] = safe_div(
            df["sensor_r"].astype(float),
            df["Obstacle_Distance_to_BLEVE_m"].astype(float)
        )

    # Safety: inf -> nan
    df = df.replace([np.inf, -np.inf], np.nan)

    # Fill numeric vs categorical safely (prevents StringArray fill error)
    num_cols = df.select_dtypes(include=[np.number]).columns
    cat_cols = df.columns.difference(num_cols)

    df[num_cols] = df[num_cols].fillna(0)
    df[cat_cols] = df[cat_cols].fillna("missing")

    return df

train_fe = fe(train)
test_fe  = fe(test)

# Auto-detect target (after sanitization)
possible_targets = ["Target_Pressure_bar", "Target_Pressure_bar_"]
TARGET = None
for t in possible_targets:
    if t in train_fe.columns:
        TARGET = t
        break

if TARGET is None:
    raise KeyError(f"Target column not found. Some cols: {train_fe.columns[:15].tolist()}")

print("Using TARGET:", TARGET)
print("FE shapes:", train_fe.shape, test_fe.shape)

In [ ]:
# =========================
# SCENARIO GROUPS (Friend's approach: EXCLUDE sensor fields)
# =========================
SENSOR_FIELDS = [
    "Sensor_ID",
    "Sensor_Position_Side",
    "Sensor_Position_x",
    "Sensor_Position_y",
    "Sensor_Position_z",
    "sensor_r",
    "sensor_r_over_obsdist",
]

exclude_for_group = set(SENSOR_FIELDS + [TARGET])
scenario_cols = [c for c in train_fe.columns if c not in exclude_for_group]

gdf = train_fe[scenario_cols].copy()

# Stabilize numeric representation
for c in gdf.columns:
    if pd.api.types.is_numeric_dtype(gdf[c]):
        gdf[c] = gdf[c].round(6)
    gdf[c] = gdf[c].astype(str)

groups = pd.util.hash_pandas_object(gdf, index=False).astype("int64").to_numpy()

print(f"unique groups: {len(set(groups))} rows: {len(groups)}")
sizes = pd.Series(groups).value_counts().values
print(f"group size min/median/max: {sizes.min()} {np.median(sizes)} {sizes.max()}")

In [ ]:
# =========================
# PREPARE X, y (CRITICAL: Use log1p not log)
# =========================
X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET].copy()
X_test = test_fe.copy()

import re

def sanitize_columns(df):
    df = df.copy()
    df.columns = [str(c) for c in df.columns]
    df.columns = [re.sub(r'[^0-9a-zA-Z_]+', '_', c) for c in df.columns]
    df.columns = [re.sub(r'_+', '_', c) for c in df.columns]
    df.columns = [c.strip('_') for c in df.columns]
    return df

X = sanitize_columns(X)
X_test = sanitize_columns(X_test)

X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Friend's key insight: log1p (log(1+x)) is more stable than log(x+eps)
y_log = np.log1p(y.values)  # log1p = log(1 + y)

print(f"X: {X.shape}, X_test: {X_test.shape}")
print(f"y range: [{y.min():.4f}, {y.max():.4f}]")
print(f"y_log range: [{y_log.min():.4f}, {y_log.max():.4f}]")

In [7]:
# =========================
# STABLE MAPE FUNCTION
# =========================
def stable_mape(y_true, y_pred, eps=1e-6):
    y_true = np.clip(np.asarray(y_true, dtype=float), eps, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), eps, None)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)))

In [ ]:
# =========================
# GROUPS + BUILD X/y + ENCODE + CV LIGHTGBM (Friend's approach) - FIXED
# =========================
def stable_mape(y_true, y_pred, eps=1e-6):
    y_true = np.clip(np.asarray(y_true, dtype=float), eps, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), eps, None)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)))

# ---- Scenario groups (exclude sensor fields) ----
SENSOR_FIELDS = [
    "Sensor_ID",
    "Sensor_Position_Side",
    "Sensor_Position_x",
    "Sensor_Position_y",
    "Sensor_Position_z",
    "sensor_r",
    "sensor_r_over_obsdist",
]
exclude_for_group = set(SENSOR_FIELDS + [TARGET])

scenario_cols = [c for c in train_fe.columns if c not in exclude_for_group]
gdf = train_fe[scenario_cols].copy()

for c in gdf.columns:
    if pd.api.types.is_numeric_dtype(gdf[c]):
        gdf[c] = gdf[c].round(6).astype(str)
    else:
        gdf[c] = gdf[c].astype(str)

groups = pd.util.hash_pandas_object(gdf, index=False).astype("int64").to_numpy()
sizes = pd.Series(groups).value_counts().values
print(f"unique groups: {len(set(groups))} rows: {len(groups)}")
print(f"group size min/median/max: {sizes.min()} {np.median(sizes)} {sizes.max()}")

# ---- y (log1p) ----
y = train_fe[TARGET].astype(float).to_numpy()
y = np.clip(y, 1e-6, None)
y_log = np.log1p(y)

# ---- X_train / X_test ----
X_train = train_fe.drop(columns=[TARGET]).copy()
X_test  = test_fe.copy()

# Align columns
common = X_train.columns.intersection(X_test.columns)
X_train = X_train[common].copy()
X_test  = X_test[common].copy()

# ---- Encode object/string columns into integer IDs ----
obj_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
for c in obj_cols:
    X_train[c] = X_train[c].fillna("NA").astype(str).str.strip().str.lower()
    X_test[c]  = X_test[c].fillna("NA").astype(str).str.strip().str.lower()

    cats = pd.Index(pd.concat([X_train[c], X_test[c]], axis=0).unique())
    mapper = {k: i for i, k in enumerate(cats)}

    X_train[c] = X_train[c].map(mapper).astype("int32")
    X_test[c]  = X_test[c].map(mapper).astype("int32")

# Final safety
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Drop constant columns
const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
if const_cols:
    X_train = X_train.drop(columns=const_cols)
    X_test  = X_test.drop(columns=const_cols)

print("X shapes:", X_train.shape, X_test.shape)
print("encoded obj cols:", len(obj_cols), "dropped constants:", len(const_cols))

# ---- LightGBM CV ----
params = {
    "objective": "regression",
    "metric": "l1",
    "boosting_type": "gbdt",

    "n_estimators": 50000,          # keep high; early stopping will stop earlier
    "learning_rate": 0.01,          # a bit higher than 0.005 so it finishes sooner
    "num_leaves": 255,
    "max_depth": -1,
    "min_data_in_leaf": 15,         # slightly higher = less overfit (often helps LB)
    "min_sum_hessian_in_leaf": 1e-3,

    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,

    "reg_alpha": 0.1,
    "reg_lambda": 2.0,

    "random_state": SEED,
    "bagging_seed": SEED,
    "feature_fraction_seed": SEED,
    "data_random_seed": SEED,

    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
    "deterministic": True,
}

N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)

test_preds_folds = []
fold_scores = []

for fold, (tr, va) in enumerate(gkf.split(X_train, y_log, groups), 1):
    print(f"--- Fold {fold} ---")

    Xtr, Xva = X_train.iloc[tr], X_train.iloc[va]
    ytr, yva = y_log[tr], y_log[va]

    model = LGBMRegressor(**params)
    model.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
    )

    pred_log = model.predict(Xva)
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 1e-6, None)

    m = stable_mape(y[va], pred)
    fold_scores.append(m)
    print(f"Fold {fold} | stable MAPE: {m:.5f} | best_iter: {model.best_iteration_}")

    test_pred_log = model.predict(X_test)
    test_pred = np.expm1(test_pred_log)
    test_pred = np.clip(test_pred, 1e-6, None)
    test_preds_folds.append(test_pred)

print("CV stable MAPE mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

# ---- Ensemble predictions ----
test_pred_final = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
test_pred_final = np.clip(test_pred_final, 1e-6, None)

# ---- Submission ----
sub = sample.copy()
target_col = sample.columns[1]
sub[target_col] = test_pred_final.astype(float)

out_path = "prediction.csv"
sub.to_csv(out_path, index=False)
print("Saved:", out_path)
sub.head()

In [ ]:
# =========================
# ENSEMBLE TEST PREDICTIONS
# =========================
test_pred_final = np.mean(np.stack(test_preds_folds, axis=0), axis=0)
test_pred_final = np.clip(test_pred_final, 0, None)

print(f"Test predictions: min={test_pred_final.min():.4f} max={test_pred_final.max():.4f} mean={test_pred_final.mean():.4f}")
print(f"Train target:     min={y.min():.4f} max={y.max():.4f} mean={y.mean():.4f}")

In [ ]:
# =========================
# BUILD SUBMISSION (MATCH SAMPLE FORMAT)
# =========================
print("sample rows:", len(sample))
print("pred rows  :", len(test_pred_final))
assert len(sample) == len(test_pred_final), "Row count mismatch"

sub = sample.copy()
target_col = sample.columns[1]
sub[target_col] = np.asarray(test_pred_final, dtype=float)

out_path = "prediction.csv"
sub.to_csv(out_path, index=False)

print(f"\nSaved to {out_path}")
print(sub.head())

In [ ]:
# Verification
check = pd.read_csv(out_path)
print("saved rows :", len(check))
print("columns   :", check.columns.tolist())
print(check.head())
print(check.tail())

In [ ]:
# =========================
# FINAL SUMMARY (FIXED)
# =========================
print("="*55)
print("FINAL SUMMARY")
print("="*55)

# fold_scores is what you actually filled during CV
for i, m in enumerate(fold_scores, 1):
    print(f"Fold {i}: MAPE={m*100:.5f}%")

print(f"{'-'*40}")
print(f"Mean CV MAPE: {np.mean(fold_scores)*100:.5f}% ± {np.std(fold_scores)*100:.5f}%")
print(f"\nExpected Kaggle: ~{np.mean(fold_scores)*100:.2f}%")
print(f"Friend's score: 68.875% (0.68875)")
print("="*55)